# Blender Render on Colab

Upload a `.blend` file on a web page and Colab's GPU renders it for you.

1. **Runtime → Change runtime type → T4 GPU** (or any GPU), then Save.
2. Set `REPO_URL` in step 1 to your copy of this repo.
3. **Runtime → Run all**. The last step prints a link to your render website.
4. Open the link, upload a `.blend` (or a `.zip` of the `.blend` plus its textures) and press **Upload & render**.

Keep this tab open while it renders: Colab stops the machine when the tab has been idle for a while, and every file on it is deleted when the runtime ends. Download your renders, or run step 5 to copy them to Google Drive.

In [ ]:
#@title 1. Settings
REPO_URL = "https://github.com/DCLEGEND2026/BlenderRender"  #@param {type:"string"}
BLENDER_VERSION = "5.2.2"  #@param ["5.2.2", "4.5.14"] {allow-input: true}
#@markdown Password for the website. Leave it empty to make a random one (it is built into the printed link).
ACCESS_KEY = ""  #@param {type:"string"}

In [ ]:
#@title 2. Install Blender and the tunnel
import os, subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none! Use Runtime > Change runtime type > GPU, or renders will be very slow")

# Libraries Blender needs on a headless Linux machine.
!apt-get -qq update > /dev/null && apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libegl1 libxkbcommon0 libsm6 libice6 > /dev/null

BLENDER_DIR = f"/content/blender-{BLENDER_VERSION}"
BLENDER_BIN = f"{BLENDER_DIR}/blender"
if not os.path.exists(BLENDER_BIN):
    series = ".".join(BLENDER_VERSION.split(".")[:2])
    url = f"https://download.blender.org/release/Blender{series}/blender-{BLENDER_VERSION}-linux-x64.tar.xz"
    print("Downloading", url)
    !wget -q --show-progress -O /content/blender.tar.xz "{url}"
    !mkdir -p "{BLENDER_DIR}" && tar -xf /content/blender.tar.xz -C "{BLENDER_DIR}" --strip-components=1 && rm /content/blender.tar.xz
!"{BLENDER_BIN}" -b --version 2>/dev/null | head -1

if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared
!pip -q install flask

In [ ]:
#@title 3. Get the website code
APP_DIR = "/content/BlenderRender"
if os.path.isdir(f"{APP_DIR}/.git"):
    !git -C "{APP_DIR}" pull -q
else:
    !git clone -q --depth 1 "{REPO_URL}" "{APP_DIR}"
assert os.path.exists(f"{APP_DIR}/app/server.py"), "Clone failed: check REPO_URL in step 1 (the repo must be public)"
print("Code ready in", APP_DIR)

In [ ]:
#@title 4. Start the website
import re, secrets, sys, time, urllib.request
from IPython.display import HTML, display

PORT = 5000
if not ACCESS_KEY:
    ACCESS_KEY = secrets.token_urlsafe(9)

# Stop a previous run of this cell.
subprocess.run(["pkill", "-f", "app/server.py"])
subprocess.run(["pkill", "-f", "cloudflared tunnel"])
time.sleep(1)

env = dict(os.environ, BLENDER_BIN=BLENDER_BIN, JOBS_DIR="/content/render_jobs",
           ACCESS_KEY=ACCESS_KEY, PORT=str(PORT))
server = subprocess.Popen([sys.executable, f"{APP_DIR}/app/server.py"], cwd=f"{APP_DIR}/app", env=env,
                          stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/healthz", timeout=2)
        break
    except Exception:
        if server.poll() is not None:
            raise SystemExit("The server stopped:\n" + open("/content/server.log").read())
        time.sleep(1)

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
public = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        public = m.group(0)
        break
    time.sleep(1)

if public:
    link = f"{public}/login?key={ACCESS_KEY}"
    display(HTML(f'<p style="font-size:16px">Render website: <a href="{link}" target="_blank">{link}</a></p>'
                 f'<p>Access key: <code>{ACCESS_KEY}</code> (anyone with the link can use this GPU, so keep it private)</p>'
                 '<p>The link can take ~20 seconds to start working.</p>'))
else:
    print("The tunnel did not start; see /content/tunnel.log. Using Colab's own link instead (works only in this browser):")
    from google.colab.output import eval_js
    link = eval_js(f"google.colab.kernel.proxyPort({PORT})") + f"login?key={ACCESS_KEY}"
    display(HTML(f'<a href="{link}" target="_blank">{link}</a>'))

In [ ]:
#@title 5. (Optional) Copy finished renders to Google Drive
import shutil
from google.colab import drive

drive.mount("/content/drive")
dest = "/content/drive/MyDrive/BlenderRender"
for job in sorted(os.listdir("/content/render_jobs")):
    out = f"/content/render_jobs/{job}/output"
    if os.path.isdir(out) and os.listdir(out):
        shutil.copytree(out, f"{dest}/{job}", dirs_exist_ok=True)
        print("Copied", job, "->", f"{dest}/{job}")

In [ ]:
#@title 6. (Optional) Show the server log
!tail -n 40 /content/server.log